# Train Hybrid LVL trên Colab: Python simulator + GPU

Mỗi worker CPU chạy một mô phỏng Python độc lập. Learner dùng GPU CUDA. Notebook tự giới hạn worker theo CPU/RAM thực được cấp; 15–20 worker chỉ hợp lý khi runtime thật sự có đủ CPU/RAM. Chạy các ô theo thứ tự. Kết quả và checkpoint được đồng bộ lên Drive định kỳ.

Notebook nhúng đúng bản mã đã dùng trong phép đo 2 giờ. Để dùng bản mã mới, đặt `SOURCE_ARCHIVE` thành đường dẫn `tar.gz` trên Drive chứa `train/` và `python-simulation/simulation/python_simulation/`, hoặc đặt `USE_GIT=True` sau khi đã push code lên `REPO_URL`.


In [ ]:
# Cấu hình một run mới. Đổi RUN_NAME khi đổi code hoặc tham số train.
RUN_NAME = "hybrid_lvl_python_r1"
REPO_URL = "https://github.com/ngocbityta/Traffic-Light-ABM.git"
REPO_REF = "main"  # Nên thay bằng commit đã kiểm tra
SOURCE_ARCHIVE = ""  # Tùy chọn: tar.gz mới trong Drive; để trống sẽ dùng bản mã nhúng bên dưới
USE_GIT = False  # True = clone REPO_URL/REPO_REF, chỉ dùng khi code đã được push
DRIVE_ROOT = "/content/drive/MyDrive/Traffic-Light-ABM/runs"
WORKERS = 0  # 0 = tự chọn; giá trị thủ công không được vượt CPU/RAM khả dụng
MAX_WORKERS = 20
TOTAL_STEPS = 1_000_000
MAX_TRAIN_SECONDS = 2 * 3600
ROLLOUT_DECISIONS = 32
WARMUP = 512
BUFFER_SIZE = 10_000
SYNC_MINUTES = 15
REWARD = "r1"


In [ ]:
import base64
EMBEDDED_SOURCE_B64 = (
    "..."
)


In [ ]:
import hashlib, json, math, os, pathlib, shutil, subprocess, sys, tarfile
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Bật Runtime > Change runtime type > GPU trước khi train"
drive.mount("/content/drive")
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
for limit_path in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    try:
        limit = int(pathlib.Path(limit_path).read_text().strip())
        if 0 < limit < ram_bytes:
            ram_bytes = limit
    except (OSError, ValueError):
        pass
ram_gb = ram_bytes / 2**30
max_by_ram = max(1, int((ram_gb - 3) // 1.5))
auto_workers = max(1, min(MAX_WORKERS, cores, max_by_ram))
workers = WORKERS or auto_workers
if workers > cores or workers > max_by_ram:
    raise ValueError(f"{workers} worker vượt CPU/RAM có thể dùng ({cores} CPU, {ram_gb:.1f} GB RAM)")
print(f"GPU: {torch.cuda.get_device_name(0)} | CPU: {cores} | RAM: {ram_gb:.1f} GB | workers: {workers}")

repo = pathlib.Path("/content/Traffic-Light-ABM")
run_dir = pathlib.Path("/content/runs") / RUN_NAME
drive_run = pathlib.Path(DRIVE_ROOT) / RUN_NAME
drive_run.mkdir(parents=True, exist_ok=True)
if USE_GIT and not SOURCE_ARCHIVE:
    if not (repo / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", REPO_REF], check=True)
    source_id = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
else:
    source_bytes = (pathlib.Path(SOURCE_ARCHIVE).read_bytes() if SOURCE_ARCHIVE
                    else base64.b64decode(EMBEDDED_SOURCE_B64))
    repo.mkdir(parents=True, exist_ok=True)
    archive_path = pathlib.Path("/content/gama_current_source.tar.gz")
    archive_path.write_bytes(source_bytes)
    subprocess.run(["tar", "-xzf", str(archive_path), "-C", str(repo)], check=True)
    source_id = hashlib.sha256(source_bytes).hexdigest()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gama-client"], check=True)
print("Source:", source_id)


In [ ]:
# Tạo cấu hình cố định để resume an toàn.
spec_path = drive_run / "run_spec.json"
spec = {"source_id": source_id, "workers": workers, "reward": REWARD,
        "total_steps": TOTAL_STEPS, "rollout_decisions": ROLLOUT_DECISIONS,
        "warmup": WARMUP, "buffer_size": BUFFER_SIZE}
if spec_path.exists():
    old = json.loads(spec_path.read_text())
    if old != spec:
        raise ValueError("Run đã có cấu hình khác. Đổi RUN_NAME hoặc dùng đúng source/tham số cũ.")
else:
    spec_path.write_text(json.dumps(spec, indent=2) + "\n")
if (drive_run / "training_state.pt").exists():
    shutil.copytree(drive_run, run_dir, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("run_spec.json", "*.tmp", "server_ws"))
    output_flag = "--resume-dir"
else:
    if run_dir.exists() and any(run_dir.iterdir()):
        raise FileExistsError(f"{run_dir} có dữ liệu cũ nhưng chưa có checkpoint")
    output_flag = "--output-dir"
run_dir.parent.mkdir(parents=True, exist_ok=True)
print("Mode:", output_flag, "| output:", run_dir)


In [ ]:
# Train 2 giờ, đồng bộ checkpoint/log mỗi SYNC_MINUTES và khi kết thúc.
import threading, time

def sync_results():
    drive_run.mkdir(parents=True, exist_ok=True)
    if not run_dir.exists():
        return
    for src in run_dir.iterdir():
        if not src.is_file() or src.name.endswith(".tmp"):
            continue
        dst = drive_run / src.name
        temp = drive_run / (src.name + ".copying")
        shutil.copy2(src, temp)
        os.replace(temp, dst)

cmd = [sys.executable, "-m", "train.logic.train",
       "--scenario", "lvl", "--backend", "python", "--algorithm", "hybrid",
       "--reward-mode", REWARD, "--device", "cuda",
       "--workers", str(workers), "--rollout-mode", "async",
       "--rollout-decisions", str(ROLLOUT_DECISIONS),
       "--total-steps", str(TOTAL_STEPS), "--episode-horizon", "8000",
       "--max-train-seconds", str(MAX_TRAIN_SECONDS),
       "--warmup", str(WARMUP), "--buffer-size", str(BUFFER_SIZE),
       "--validate-every", "50", "--checkpoint-every", "5",
       output_flag, str(run_dir)]
env = os.environ.copy()
env["PYTHONPATH"] = str(repo) + os.pathsep + str(repo / "python-simulation" / "simulation")
stop_sync = threading.Event()
def sync_loop():
    while not stop_sync.wait(SYNC_MINUTES * 60):
        try:
            sync_results()
            print("Đã đồng bộ lên Drive", flush=True)
        except Exception as exc:
            print("Đồng bộ tạm lỗi:", exc, flush=True)
thread = threading.Thread(target=sync_loop, daemon=True)
thread.start()
started = time.monotonic()
try:
    with (run_dir.parent / f"{RUN_NAME}.console.log").open("a") as log:
        process = subprocess.Popen(cmd, cwd=repo, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        rc = process.wait()
finally:
    stop_sync.set()
    thread.join(timeout=5)
    sync_results()
print(f"Exit: {rc} | thời gian: {(time.monotonic()-started)/3600:.2f} giờ")
if rc:
    raise RuntimeError("Train dừng với lỗi; xem log phía trên")


In [ ]:
# Tóm tắt tốc độ và số bước sau khi train.
import csv
rows = list(csv.DictReader((run_dir / "training_log.csv").open()))
steps = sum(int(row["decisions"]) for row in rows)
summary = json.loads((run_dir / "training_summary.json").read_text())
print(f"Decisions: {steps:,} | rollouts: {len(rows):,} | steps/giờ: {steps / (summary['train_wall_seconds']/3600):,.0f}")
print("Kết quả:", drive_run)
